In [7]:
import pdfplumber
import os

pdf_path = r"C:\Users\Asus\Documents\Oulu Research Intern\IPCC AR6 WG1 The Physical Science Basis\IPCC_AR6_WGI_SPM.pdf"

# Extract text page by page
pages = []
with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        pages.append({
            "page_number": i + 1,
            "text": text
        })

print(f"Total pages extracted: {len(pages)}")
print(f"\n--- PAGE 1 PREVIEW ---\n")
print(pages[0]["text"][:1000])

Total pages extracted: 32

--- PAGE 1 PREVIEW ---

Summary for
Policymakers


In [8]:
# How many pages total
print(f"Total pages: {len(pages)}")

# Look at pages 2 and 3 as well
for i in [1, 2, 3]:
    print(f"\n--- PAGE {i+1} ---")
    print(pages[i]["text"][:500])

Total pages: 32

--- PAGE 2 ---


--- PAGE 3 ---
Summary for
Policymakers
Drafting Authors:
Richard P. Allan (United Kingdom), Paola A. Arias (Colombia), Sophie Berger (France/Belgium), Josep G.
Canadell (Australia), Christophe Cassou (France), Deliang Chen (Sweden), Annalisa Cherchi (Italy), Sarah
SPM L. Connors (France/United Kingdom), Erika Coppola (Italy), Faye Abigail Cruz (Philippines), Aïda Diongue-
Niang (Senegal), Francisco J. Doblas-Reyes (Spain), Hervé Douville (France), Fatima Driouech (Morocco),
Tamsin L. Edwards (United Kingdom),

--- PAGE 4 ---
Summary for Policymakers
Introduction
This Summary for Policymakers (SPM) presents key findings of the Working Group I (WGI) contribution to the Intergovernmental
Panel on Climate Change (IPCC) Sixth Assessment Report (AR6)1 on the physical science basis of climate change. The report builds
upon the 2013 Working Group I contribution to the IPCC’s Fifth Assessment Report (AR5) and the 2018–2019 IPCC Special Reports2
of the AR6 cycl

In [9]:
# Find pages where text extraction failed or returned very little text
problem_pages = []
for page in pages:
    if page["text"] is None:
        problem_pages.append((page["page_number"], "None - no text extracted"))
    elif len(page["text"]) < 100:
        problem_pages.append((page["page_number"], f"Very short - only {len(page['text'])} characters"))

print(f"Total problem pages: {len(problem_pages)}")
for page_num, issue in problem_pages:
    print(f"Page {page_num}: {issue}")

Total problem pages: 3
Page 1: Very short - only 24 characters
Page 2: Very short - only 0 characters
Page 32: Very short - only 0 characters


In [10]:
# Basic stats
total_chars = sum(len(p["text"]) for p in pages if p["text"])
avg_chars = total_chars // len(pages)

print(f"Total characters extracted: {total_chars}")
print(f"Average characters per page: {avg_chars}")
print(f"Estimated total words: {total_chars // 5}")

Total characters extracted: 122789
Average characters per page: 3837
Estimated total words: 24557


In [11]:
import re

def clean_text(text):
    if text is None:
        return ""
    
    # Remove excessive whitespace and blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # Remove standalone page numbers (a number alone on a line)
    text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)
    
    # Remove lines that are just dashes or underscores
    text = re.sub(r'^[-_]+$', '', text, flags=re.MULTILINE)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    return text

# Apply cleaning to all pages
cleaned_pages = []
for page in pages:
    cleaned_text = clean_text(page["text"])
    cleaned_pages.append({
        "page_number": page["page_number"],
        "text": cleaned_text
    })

# Check the result on page 1
print("--- CLEANED PAGE 1 ---")
print(cleaned_pages[5]["text"][:1000])

--- CLEANED PAGE 1 ---
Summary for Policymakers
A.1.8 Changes in the land biosphere since 1970 are consistent with global warming: climate zones have shifted poleward in
both hemispheres, and the growing season has on average lengthened by up to two days per decade since the 1950s
in the Northern Hemisphere extratropics (high confidence).
{2.3, TS.2.6}
SPM
Human influence has warmed the climate at a rate that is unprecedented
in at least the last 2000 years
Changes in global surface temperature relative to 1850–1900
(a) Change in global surface temperature (decadal average) (b) Change in global surface temperature (annual average) as observed and
as reconstructed (1–2000) and observed (1850–2020) simulated using human & natural and only natural factors (both 1850–2020)
ºC ºC
2.0 2.0
Warming is unprecedented
in more than 2000 years
1.5 1.5
Warmest multi-century observed
period in more than
100,000 years simulated
1.0 1.0 1.0 human &
observed natural
0.5 0.5
0.2 simulated
natural only
0.

In [12]:
import os

# Create the processed folder if it does not exist
os.makedirs(r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\data\processed", exist_ok=True)

# Create output path
output_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\data\processed\IPCC_AR6_WGI_SPM.txt"

# Combine all cleaned pages into one text with page markers
full_text = ""
for page in cleaned_pages:
    full_text += f"\n\n--- PAGE {page['page_number']} ---\n\n"
    full_text += page["text"]

# Save to file
with open(output_path, "w", encoding="utf-8") as f:
    f.write(full_text)

print(f"Saved successfully")
print(f"Total characters saved: {len(full_text)}")

Saved successfully
Total characters saved: 123207


In [13]:
import pdfplumber
import re
import os

def extract_and_clean_pdf(pdf_path):
    """Extract and clean text from a single PDF file"""
    pages = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            pages.append({
                "page_number": i + 1,
                "text": text
            })
    
    # Clean each page
    cleaned_pages = []
    for page in pages:
        cleaned_text = clean_text(page["text"])
        cleaned_pages.append({
            "page_number": page["page_number"],
            "text": cleaned_text
        })
    
    # Combine all pages into one text
    full_text = ""
    for page in cleaned_pages:
        full_text += f"\n\n--- PAGE {page['page_number']} ---\n\n"
        full_text += page["text"]
    
    return full_text, len(cleaned_pages)


def process_all_pdfs(input_folder, output_folder):
    """Process all PDFs in input folder and save to output folder"""
    
    # Create output folder if it does not exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all PDF files
    pdf_files = [f for f in os.listdir(input_folder) if f.endswith(".pdf")]
    
    print(f"Found {len(pdf_files)} PDF files to process\n")
    
    results = []
    
    for pdf_file in pdf_files:
        pdf_path = os.path.join(input_folder, pdf_file)
        output_filename = pdf_file.replace(".pdf", ".txt")
        output_path = os.path.join(output_folder, output_filename)
        
        # Skip if already processed
        if os.path.exists(output_path):
            print(f"Skipping {pdf_file} — already processed")
            continue
        
        print(f"Processing: {pdf_file}")
        
        try:
            full_text, num_pages = extract_and_clean_pdf(pdf_path)
            
            # Save to file
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(full_text)
            
            word_count = len(full_text) // 5
            print(f"Done — {num_pages} pages, ~{word_count} words")
            
            results.append({
                "filename": pdf_file,
                "pages": num_pages,
                "words": word_count,
                "status": "success"
            })
            
        except Exception as e:
            print(f"Failed: {pdf_file} — {e}")
            results.append({
                "filename": pdf_file,
                "status": "failed",
                "error": str(e)
            })
    
    print(f"\nCompleted. {len([r for r in results if r['status'] == 'success'])} succeeded, {len([r for r in results if r['status'] == 'failed'])} failed")
    return results


# Run it — using correct Capital D paths
input_folder = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\raw"
output_folder = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\processed"

results = process_all_pdfs(input_folder, output_folder)

Found 6 PDF files to process

Skipping IPCC_AR6_WGIII_SummaryForPolicymakers.pdf — already processed
Skipping IPCC_AR6_WGIII_TechnicalSummary.pdf — already processed
Skipping IPCC_AR6_WGII_SummaryForPolicymakers.pdf — already processed
Skipping IPCC_AR6_WGII_TechnicalSummary.pdf — already processed
Skipping IPCC_AR6_WGI_SPM.pdf — already processed
Skipping IPCC_AR6_WGI_TS.pdf — already processed

Completed. 0 succeeded, 0 failed
